# Container Deployment and Runtime Setup
Companion notebook for the **Container Deployment and Runtime Setup** section of Chapter 12.

Set runtime values through environment variables, not hardcoded values -- the book's example settings:

In [1]:
runtime_env = {
    "PORT": "8080",
    "MODEL_NAME": "gpt-4o-mini",
    "TOOL_TIMEOUT_MS": "8000",
    "REDIS_URL": "redis://cache:6379",
    "LOG_LEVEL": "info",
}
for key, value in runtime_env.items():
    print(f"{key}={value}")

PORT=8080
MODEL_NAME=gpt-4o-mini
TOOL_TIMEOUT_MS=8000
REDIS_URL=redis://cache:6379
LOG_LEVEL=info


This is the book's health-check app, unchanged. It actually runs and is tested below with FastAPI's `TestClient` -- no server process or open port needed:

In [2]:
from fastapi import FastAPI

app = FastAPI()

ready = False


@app.on_event("startup")
async def startup():
    global ready
    # load prompts, warm cache, create clients
    ready = True


@app.get("/healthz")
def healthz():
    return {"ok": True}


@app.get("/readyz")
def readyz():
    return {"ready": ready}

C:\Users\himan\AppData\Local\Temp\ipykernel_44356\2174396696.py:8: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


In [3]:
from fastapi.testclient import TestClient

with TestClient(app) as client:  # entering the context runs FastAPI's startup event
    print("healthz:", client.get("/healthz").json())
    print("readyz: ", client.get("/readyz").json())

healthz: {'ok': True}
readyz:  {'ready': True}


C:\Users\himan\AppData\Local\Temp\claude\C--Users-himan\eec66fed-60c1-4f01-b9be-916a12ead6c3\scratchpad\nb_build\testenv7\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


One thing worth knowing: `@app.on_event("startup")` above works, but current FastAPI marks it deprecated in favor of a **lifespan** context manager -- the modern equivalent, showing the readiness point the book makes explicitly: readiness should confirm real dependencies are usable, not just "the app object exists":

In [4]:
from contextlib import asynccontextmanager

dependency_state = {"model_client": False, "vector_store": False}


@asynccontextmanager
async def lifespan(app: FastAPI):
    # warm-up: create the model client, connect the vector store -- readiness depends on both
    dependency_state["model_client"] = True
    dependency_state["vector_store"] = True
    yield
    dependency_state["model_client"] = False
    dependency_state["vector_store"] = False


app_v2 = FastAPI(lifespan=lifespan)


@app_v2.get("/healthz")
def healthz_v2():
    return {"ok": True}  # process is alive, regardless of dependency state


@app_v2.get("/readyz")
def readyz_v2():
    all_ready = all(dependency_state.values())
    return {"ready": all_ready, "dependencies": dependency_state}

In [5]:
with TestClient(app_v2) as client:
    print("healthz:", client.get("/healthz").json())
    print("readyz: ", client.get("/readyz").json())

healthz: {'ok': True}
readyz:  {'ready': True, 'dependencies': {'model_client': True, 'vector_store': True}}


A pod that's *alive* but still warming caches should not receive traffic yet -- that's exactly the gap between `/healthz` (liveness) and `/readyz` (readiness) above. Log every request with one correlation ID plus graph step names, tool calls, latency, and final status -- in a micro-graph setup, that's what lets you follow one request across containers.